# Week 7 – Model Optimisation & Trade-offs
## Building and Benchmarking Complex Models Against the Week 6 Baseline

**Student:** Shari Oliver

**Programme:** CariSurg MedTech Pathways — Healthcare AI

**Dataset:** `yaleemmlc_admissionprediction_triage.csv`

**Submission stage:** Final Submission (Tuesday 11:59 p.m. AST)

**Random seed:** 42 (fixed throughout — set once at the top of this notebook and reused for the train/test split and every model)

---

## Purpose of This Notebook

Week 6 delivered two interpretable baseline classifiers — logistic regression and a bounded decision tree — for predicting a patient's Emergency Severity Index (ESI) at triage. The ED Board then asked a harder question: **what does a more sophisticated model buy us, and is it worth the extra complexity?**

This notebook now compares **three** complex model candidates against the Week 6 baseline, not just one:

1. **Random Forest**
2. **XGBoost** (gradient boosting)
3. **LightGBM** (gradient boosting)

All three are trained on the identical Week 6 feature set and train/test split, benchmarked on the same six quantitative axes (accuracy, precision, recall, F1, training time, inference time), and assessed on the same qualitative interpretability axis. A best-of-three selection step then picks the strongest complex-model candidate — on the primary clinical metric, ESI Level 1 recall — before that candidate is compared head-to-head against the Week 6 baseline in the cost-benefit memo.

**How to run this notebook:** place `yaleemmlc_admissionprediction_triage.csv` in the same location you used for Week 6. This notebook also needs `xgboost` and `lightgbm` installed, which are not part of the standard scikit-learn stack — run the install cell below first if either import fails.

**Note on results already in the docs:** the cost-benefit memo and decision journal already committed to `/docs` reflect the Random Forest vs. baseline comparison only, run before XGBoost and LightGBM were added. Once you re-run this notebook end-to-end with the two new models, update Section 18 with the real numbers, then let me know so the memo, benchmark table and decision journal can be revised to reflect whichever of the three complex models actually wins.

## 1. Import Libraries and Configure the Environment

Same as before, plus `XGBClassifier` and `LGBMClassifier`. Both are gradient-boosting libraries, not part of the core scikit-learn install — run the cell below first if you don't already have them.

In [ ]:
# Run once if xgboost / lightgbm are not already installed.
# %pip install xgboost lightgbm --quiet

In [ ]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score
)

warnings.filterwarnings("ignore")

RANDOM_SEED = 42

print("Random Seed =", RANDOM_SEED)

## 2. Create Output Folders

Generated figures and summary tables are saved into a Week 7 output folder, mirroring the Week 6 structure, so they can be copied straight into the GitHub `\notebooks` and `\docs` folders.

In [ ]:
OUTPUT_DIR = Path("week7_outputs")

PLOT_DIR = OUTPUT_DIR / "plots"
DOC_DIR = OUTPUT_DIR / "docs"

PLOT_DIR.mkdir(parents=True, exist_ok=True)
DOC_DIR.mkdir(parents=True, exist_ok=True)

print("Plots:", PLOT_DIR)
print("Documents:", DOC_DIR)

## 3. Load the Dataset

Same loading logic as Week 6. The raw dataset is **not committed to GitHub** because it contains programme-controlled data. The notebook checks the common local, Colab and Google Drive locations before loading.

In [ ]:
DATASET_NAME = "yaleemmlc_admissionprediction_triage.csv"

possible_paths = [
    Path(DATASET_NAME),
    Path("/content") / DATASET_NAME,
    Path("/content/drive/MyDrive") / DATASET_NAME,
    Path("/content/drive/MyDrive/CariSurg") / DATASET_NAME,
]

DATA_PATH = None

for path in possible_paths:
    if path.exists():
        DATA_PATH = path
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not locate '{DATASET_NAME}'. "
        "Please upload the dataset or update the file path."
    )

df = pd.read_csv(DATA_PATH, index_col=0)

print("Dataset loaded successfully!")
print(f"Dataset location: {DATA_PATH}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

df.head()

## 4. Recreate the Week 6 Modelling Dataset

To make Week 7 a fair test of *model complexity* rather than a test of *different data*, the target definition and feature selection logic are copied verbatim from the Week 6 notebook. The predictor set is restricted to information realistically available at the moment of triage (vital signs, arrival information, chief complaint flags, and age), with any column that looks like a post-triage outcome (admission, disposition, length of stay, mortality, ICU, etc.) deliberately excluded to avoid data leakage.

**Note on fairness:** columns such as gender, ethnicity, race, language, religion, marital status, employment status and insurance are present in the raw dataset but were *not* used as predictors in Week 6, and are not used here either. This keeps the model from learning to triage patients differently based on demographic or socioeconomic attributes.

In [ ]:
# -----------------------------
# Define target variable
# -----------------------------

TARGET = "esi"

# -----------------------------
# Candidate feature groups (identical logic to Week 6)
# -----------------------------

vital_features = [
    col for col in df.columns
    if any(term in col.lower() for term in
           ["hr","sbp","dbp","rr","o2","temp","glucose"])
]

arrival_features = [
    col for col in df.columns
    if "arrival" in col.lower()
]

chief_complaint_features = [
    col for col in df.columns
    if col.lower().startswith("cc_")
]

demographic_features = [
    col for col in ["age"]
    if col in df.columns
]

# Build final predictor list

selected_features = (
    demographic_features +
    vital_features +
    arrival_features +
    chief_complaint_features
)

selected_features = list(dict.fromkeys(selected_features))

print("Number of selected features:", len(selected_features))
print(selected_features[:20])

In [ ]:
# -----------------------------
# Leakage check (identical logic to Week 6, kept as a guardrail)
# -----------------------------

leakage_keywords = [
    "admit",
    "admission",
    "disposition",
    "hospital",
    "los",
    "outcome",
    "death",
    "mortality",
    "icu"
]

possible_leakage = []

for col in df.columns:
    if any(word in col.lower() for word in leakage_keywords):
        possible_leakage.append(col)

print("Potential leakage variables (confirmed excluded from selected_features):")
possible_leakage

In [ ]:
model_df = df[selected_features + [TARGET]].copy()

# Remove rows with missing target values
model_df = model_df.dropna(subset=[TARGET])

# Ensure target is numeric
model_df[TARGET] = pd.to_numeric(
    model_df[TARGET],
    errors="coerce"
)

model_df = model_df.dropna(subset=[TARGET])

model_df[TARGET] = model_df[TARGET].astype(int)

X = model_df[selected_features]
y = model_df[TARGET]

print("Predictor matrix shape:", X.shape)
print("Target vector shape:", y.shape)

## 5. Recreate the Week 6 Train/Test Split

Same 80/20 stratified split, same `random_state=42`. Because the feature selection above is byte-for-byte identical to Week 6 and the seed is unchanged, this reproduces the *exact same* train/test partition used for the baseline models.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED
)

print("Training observations:", X_train.shape[0])
print("Testing observations:", X_test.shape[0])

## 6. Preprocessing Pipelines

Identical to Week 6: numeric features are median-imputed, categorical features are most-frequent-imputed and one-hot encoded. Logistic regression additionally gets standardised numeric features. Random Forest, XGBoost and LightGBM, like the decision tree, do not need feature scaling, so they all reuse the tree preprocessing pipeline.

In [ ]:
numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numeric Features:", len(numeric_features))
print("Categorical Features:", len(categorical_features))

In [ ]:
numeric_pipeline_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_pipeline_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_lr = ColumnTransformer([
    ("numeric", numeric_pipeline_lr, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

preprocessor_tree = ColumnTransformer([
    ("numeric", numeric_pipeline_tree, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

## 7. Re-Establish the Week 6 Baselines, With Timing Instrumentation

Week 6 measured accuracy and classification metrics but did not measure **training time** or **inference time**. Both are retrained here (same hyperparameters, same seed, same split as Week 6) so timing can be captured under identical, controlled conditions alongside every complex model tested below.

`fit_and_time()` and `predict_and_time()` are used for every model in this notebook, baseline and complex, so the timing methodology stays consistent.

In [ ]:
def fit_and_time(pipeline, X_train, y_train, sample_weight=None):
    """
    Fit a pipeline and return (fitted_pipeline, training_time_seconds).
    sample_weight is passed through to the final estimator's fit() call
    when the model itself does not support class_weight natively
    (e.g. XGBoost).
    """
    start = time.perf_counter()
    if sample_weight is not None:
        pipeline.fit(X_train, y_train, model__sample_weight=sample_weight)
    else:
        pipeline.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    return pipeline, elapsed


def predict_and_time(pipeline, X_test, n_single_row_samples=200):
    """
    Predict on the full test set and return:
      - predictions
      - batch inference time per prediction (ms) = full-batch predict time / n_test
      - single-row inference time per prediction (ms) = mean of timing
        individual .predict() calls on a sample of rows
    """
    start = time.perf_counter()
    predictions = pipeline.predict(X_test)
    batch_elapsed = time.perf_counter() - start
    batch_ms_per_prediction = (batch_elapsed / len(X_test)) * 1000

    sample_n = min(n_single_row_samples, len(X_test))
    sample_idx = np.random.RandomState(RANDOM_SEED).choice(
        len(X_test), size=sample_n, replace=False
    )

    single_row_times = []
    for i in sample_idx:
        row = X_test.iloc[[i]]
        start = time.perf_counter()
        pipeline.predict(row)
        single_row_times.append(time.perf_counter() - start)

    single_row_ms_per_prediction = np.mean(single_row_times) * 1000

    return predictions, batch_ms_per_prediction, single_row_ms_per_prediction

In [ ]:
# -----------------------------
# Logistic Regression (Week 6 baseline #1)
# -----------------------------

logistic_model = Pipeline([
    ("preprocessor", preprocessor_lr),
    ("model",
     LogisticRegression(
         max_iter=2000,
         class_weight="balanced",
         random_state=RANDOM_SEED
     ))
])

logistic_model, logistic_train_time = fit_and_time(
    logistic_model, X_train, y_train
)

(
    logistic_predictions,
    logistic_batch_ms,
    logistic_single_ms
) = predict_and_time(logistic_model, X_test)

print("Logistic Regression trained successfully.")
print(f"Training time: {logistic_train_time:.3f} s")
print(f"Inference time (single-row): {logistic_single_ms:.4f} ms/prediction")

In [ ]:
# -----------------------------
# Decision Tree (Week 6 baseline #2)
# -----------------------------

decision_tree_model = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model",
     DecisionTreeClassifier(
         max_depth=5,
         class_weight="balanced",
         random_state=RANDOM_SEED
     ))
])

decision_tree_model, tree_train_time = fit_and_time(
    decision_tree_model, X_train, y_train
)

(
    tree_predictions,
    tree_batch_ms,
    tree_single_ms
) = predict_and_time(decision_tree_model, X_test)

print("Decision Tree trained successfully.")
print(f"Training time: {tree_train_time:.3f} s")
print(f"Inference time (single-row): {tree_single_ms:.4f} ms/prediction")

## 8. Choosing the Complex Models: Random Forest, XGBoost and LightGBM

Rather than committing to one complex-model family up front, this final submission tests all three tree-ensemble options the brief allows (excluding the small MLP, ruled out early for needing more training data and offering no native feature-importance output) and lets the benchmark numbers decide:

1. **Random Forest** — bagged trees, trained independently and averaged. Simple to reason about, no boosting-specific tuning required.
2. **XGBoost** — gradient-boosted trees, trained sequentially, each tree correcting the previous ensemble's errors. Typically stronger than Random Forest on tabular data, at the cost of more hyperparameters and a class-imbalance handling mechanism that needs to be set up manually for multiclass problems (via `sample_weight`, since XGBoost's classifier has no direct multiclass `class_weight` argument).
3. **LightGBM** — a second gradient-boosting implementation, generally faster to train than XGBoost on larger datasets due to its histogram-based splitting, and it does support `class_weight="balanced"` directly for multiclass problems.

**Hyperparameters used for this run** (a reasonable starting point for all three, not yet tuned):
- `n_estimators=300`, `random_state=42` for all three
- Random Forest: `max_depth=10`, `class_weight="balanced"`, `n_jobs=-1`
- XGBoost: `max_depth=6` (XGBoost's own default depth), `learning_rate=0.1`, `sample_weight` computed via `compute_sample_weight(class_weight="balanced", y=y_train)` since multiclass `class_weight` is not natively supported
- LightGBM: `max_depth=-1` (LightGBM default, unbounded but regularised via `num_leaves`), `learning_rate=0.1`, `class_weight="balanced"`

**Selection rule:** after all three are benchmarked below, the model with the **highest ESI Level 1 recall** is selected as the Week 7 complex-model recommendation, since that has been this project's primary clinical metric since Week 6. Training time and inference time are used as tie-breakers if two models are within 0.02 of each other on ESI 1 recall.

In [ ]:
# -----------------------------
# Random Forest (complex model candidate #1)
# -----------------------------

random_forest_model = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model",
     RandomForestClassifier(
         n_estimators=300,
         max_depth=10,
         class_weight="balanced",
         random_state=RANDOM_SEED,
         n_jobs=-1
     ))
])

random_forest_model, rf_train_time = fit_and_time(
    random_forest_model, X_train, y_train
)

(
    rf_predictions,
    rf_batch_ms,
    rf_single_ms
) = predict_and_time(random_forest_model, X_test)

print("Random Forest trained successfully.")
print(f"Training time: {rf_train_time:.3f} s")
print(f"Inference time (single-row): {rf_single_ms:.4f} ms/prediction")

In [ ]:
# -----------------------------
# XGBoost (complex model candidate #2)
# -----------------------------
# XGBoost's classifier has no direct multiclass class_weight argument,
# so per-sample weights are computed manually and passed through fit().

xgb_sample_weight = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

xgboost_model = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model",
     XGBClassifier(
         n_estimators=300,
         max_depth=6,
         learning_rate=0.1,
         random_state=RANDOM_SEED,
         eval_metric="mlogloss",
         n_jobs=-1
     ))
])

xgboost_model, xgb_train_time = fit_and_time(
    xgboost_model, X_train, y_train, sample_weight=xgb_sample_weight
)

(
    xgb_predictions,
    xgb_batch_ms,
    xgb_single_ms
) = predict_and_time(xgboost_model, X_test)

print("XGBoost trained successfully.")
print(f"Training time: {xgb_train_time:.3f} s")
print(f"Inference time (single-row): {xgb_single_ms:.4f} ms/prediction")

In [ ]:
# -----------------------------
# LightGBM (complex model candidate #3)
# -----------------------------

lightgbm_model = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("model",
     LGBMClassifier(
         n_estimators=300,
         learning_rate=0.1,
         class_weight="balanced",
         random_state=RANDOM_SEED,
         n_jobs=-1,
         verbose=-1
     ))
])

lightgbm_model, lgbm_train_time = fit_and_time(
    lightgbm_model, X_train, y_train
)

(
    lgbm_predictions,
    lgbm_batch_ms,
    lgbm_single_ms
) = predict_and_time(lightgbm_model, X_test)

print("LightGBM trained successfully.")
print(f"Training time: {lgbm_train_time:.3f} s")
print(f"Inference time (single-row): {lgbm_single_ms:.4f} ms/prediction")

## 9. Six-Axis Quantitative Benchmark — All Five Models

Every model — both Week 6 baselines and all three Week 7 complex-model candidates — is evaluated on the same six axes: accuracy, precision, recall, F1 (macro and weighted, plus the ESI Level 1 class specifically), training time and inference time. `evaluate_model()` is unchanged from Week 6; the loop below runs it once per model so the table stays easy to extend if a fourth candidate is ever added.

In [ ]:
def evaluate_model(model_name, y_true, y_pred):
    """
    Create summary and per-class evaluation tables
    for one classification model. (Unchanged from Week 6.)
    """
    report = classification_report(
        y_true,
        y_pred,
        output_dict=True,
        zero_division=0
    )

    summary = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (macro)": report["macro avg"]["precision"],
        "Recall (macro)": report["macro avg"]["recall"],
        "Macro F1": report["macro avg"]["f1-score"],
        "Weighted F1": report["weighted avg"]["f1-score"],
        "ESI 1 Precision": report.get("1", {}).get("precision", np.nan),
        "ESI 1 Recall": report.get("1", {}).get("recall", np.nan),
        "ESI 1 F1": report.get("1", {}).get("f1-score", np.nan)
    }

    per_class = pd.DataFrame(report).T.reset_index()
    per_class = per_class.rename(
        columns={"index": "Class or Average"}
    )
    per_class.insert(0, "Model", model_name)

    return summary, per_class

In [ ]:
model_results = {
    "Logistic Regression (Week 6)": {
        "predictions": logistic_predictions,
        "train_time": logistic_train_time,
        "single_ms": logistic_single_ms,
    },
    "Decision Tree (Week 6)": {
        "predictions": tree_predictions,
        "train_time": tree_train_time,
        "single_ms": tree_single_ms,
    },
    "Random Forest (Week 7)": {
        "predictions": rf_predictions,
        "train_time": rf_train_time,
        "single_ms": rf_single_ms,
    },
    "XGBoost (Week 7)": {
        "predictions": xgb_predictions,
        "train_time": xgb_train_time,
        "single_ms": xgb_single_ms,
    },
    "LightGBM (Week 7)": {
        "predictions": lgbm_predictions,
        "train_time": lgbm_train_time,
        "single_ms": lgbm_single_ms,
    },
}

summary_rows = []
per_class_tables = []

for model_name, result in model_results.items():
    summary, per_class = evaluate_model(
        model_name,
        y_test,
        result["predictions"]
    )
    summary["Training Time (s)"] = result["train_time"]
    summary["Inference Time — Single Row (ms/prediction)"] = result["single_ms"]

    summary_rows.append(summary)
    per_class_tables.append(per_class)

six_axis_benchmark = pd.DataFrame(summary_rows).round(4)
per_class_metrics = pd.concat(per_class_tables, ignore_index=True).round(3)

display(six_axis_benchmark)

In [ ]:
display(per_class_metrics)

In [ ]:
six_axis_benchmark.to_csv(
    DOC_DIR / "SOliver_Week7_Final_Benchmark_Table.csv",
    index=False
)

per_class_metrics.to_csv(
    DOC_DIR / "SOliver_Week7_Per_Class_Metrics.csv",
    index=False
)

print("Final benchmark table and per-class metrics saved successfully.")

## 10. Selecting the Best Complex Model

Per the selection rule in Section 8: the complex-model candidate (Random Forest, XGBoost or LightGBM) with the **highest ESI Level 1 recall** is chosen as the Week 7 recommendation, with training time and inference time as tie-breakers if two candidates are within 0.02 recall of each other.

In [ ]:
complex_model_names = ["Random Forest (Week 7)", "XGBoost (Week 7)", "LightGBM (Week 7)"]

complex_candidates = six_axis_benchmark[
    six_axis_benchmark["Model"].isin(complex_model_names)
].sort_values("ESI 1 Recall", ascending=False).reset_index(drop=True)

display(complex_candidates)

best_recall = complex_candidates.loc[0, "ESI 1 Recall"]
within_tolerance = complex_candidates[
    complex_candidates["ESI 1 Recall"] >= best_recall - 0.02
]

if len(within_tolerance) > 1:
    # Tie-break on inference time, then training time
    best_complex_model = within_tolerance.sort_values(
        ["Inference Time — Single Row (ms/prediction)", "Training Time (s)"]
    ).iloc[0]["Model"]
    print(f"Multiple candidates within 0.02 ESI 1 recall of each other — "
          f"tie-broken on inference/training time.")
else:
    best_complex_model = complex_candidates.iloc[0]["Model"]

print(f"\nBest complex model (by ESI 1 recall, tie-broken on cost): {best_complex_model}")

## 11. Interpretability — the Seventh, Qualitative Axis

The brief requires a qualitative interpretability axis: **can you explain a single prediction to Dr Reyes in under a minute?**

- **Logistic Regression:** Yes, in principle — each feature has a signed coefficient — but with dozens of one-hot encoded chief-complaint columns, explaining one specific patient's prediction means walking through many small contributions at once.
- **Decision Tree (`max_depth=5`):** Yes, easily. A single prediction is one path down the tree — at most 5 yes/no questions.
- **Random Forest / XGBoost / LightGBM:** Not directly, for any of the three. There is no single path to point to — a prediction is an aggregate across hundreds of trees (bagged for Random Forest, boosted sequentially for XGBoost and LightGBM). What is available for all three is a **global** feature importance ranking; **local**, per-patient explanations would need SHAP (or permutation importance on a single row) for any of them — this is not yet built for any of the three ensemble models.

Global feature importance is computed below for whichever model Section 10 selected as the best complex candidate, so the plot reflects the model actually being recommended rather than an arbitrary one.

In [ ]:
# Map model names to their fitted pipelines, so the best model's
# feature importances can be pulled dynamically.

fitted_pipelines = {
    "Random Forest (Week 7)": random_forest_model,
    "XGBoost (Week 7)": xgboost_model,
    "LightGBM (Week 7)": lightgbm_model,
}

best_pipeline = fitted_pipelines[best_complex_model]

best_preprocessor = best_pipeline.named_steps["preprocessor"]
best_feature_names = best_preprocessor.get_feature_names_out()

best_importances = pd.DataFrame({
    "Feature": best_feature_names,
    "Importance": best_pipeline.named_steps["model"].feature_importances_
}).sort_values("Importance", ascending=False)

top_features = best_importances.head(15)

display(top_features)

In [ ]:
plt.figure(figsize=(9, 7))

sns.barplot(
    data=top_features,
    y="Feature",
    x="Importance",
    palette="viridis"
)

plt.title(f"{best_complex_model} — Top 15 Feature Importances")
plt.xlabel("Relative Importance")
plt.ylabel("")

plt.tight_layout()

plt.savefig(
    PLOT_DIR / "SOliver_Week7_BestComplexModel_Feature_Importance.png",
    dpi=300
)

plt.show()

### Interpretation

The top features give a global sense of what the selected complex model relies on. This should be sense-checked against clinical intuition before the memo is finalised: if the top features look clinically implausible, that is a red flag worth raising with Dr Reyes, not a result to quietly accept.

**Still to do:** SHAP values for a small set of individual test patients (including at least one true ESI Level 1 case), so the memo can show — not just claim — what "explaining a prediction" would look like for whichever ensemble model is recommended.

## 12. Confusion Matrices — All Three Complex Models

Same confusion matrix approach as Week 6, applied to all three complex-model candidates so they can be compared side by side. Rows are actual ESI level, columns are predicted ESI level. As in Week 6, the cell of greatest concern is any true ESI Level 1 patient predicted as a less urgent level.

In [ ]:
labels = sorted(y_test.unique())

def save_confusion_matrix(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    display_matrix = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=labels
    )

    fig, ax = plt.subplots(figsize=(7, 6))

    display_matrix.plot(
        ax=ax,
        cmap="Blues",
        values_format="d",
        colorbar=False
    )

    ax.set_title(title)
    ax.set_xlabel("Predicted ESI Level")
    ax.set_ylabel("Actual ESI Level")

    plt.tight_layout()

    output_path = PLOT_DIR / filename
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_path)

    return cm


confusion_matrices = {}

for model_name, predictions_key in [
    ("Random Forest (Week 7)", rf_predictions),
    ("XGBoost (Week 7)", xgb_predictions),
    ("LightGBM (Week 7)", lgbm_predictions),
]:
    safe_name = model_name.split(" (")[0].replace(" ", "")
    confusion_matrices[model_name] = save_confusion_matrix(
        y_test,
        predictions_key,
        f"{model_name} Confusion Matrix",
        f"SOliver_Week7_{safe_name}_Confusion_Matrix.png"
    )

## 13. ESI Level 1 Failure-Mode Analysis — All Models

Same methodology as Week 6: for every true ESI Level 1 patient, check whether each model correctly identified them, and if not, where they were misclassified to. This stays the primary safety lens for judging whether added complexity is actually worth it.

In [ ]:
def esi1_failure_summary(y_true, y_pred, model_name):
    results = pd.DataFrame({
        "Actual ESI": y_true.to_numpy(),
        "Predicted ESI": y_pred
    })

    actual_esi1 = results[results["Actual ESI"] == 1].copy()
    missed_esi1 = actual_esi1[actual_esi1["Predicted ESI"] != 1]

    summary = pd.DataFrame({
        "Model": [model_name],
        "Actual ESI 1 Patients": [len(actual_esi1)],
        "Correctly Identified": [(actual_esi1["Predicted ESI"] == 1).sum()],
        "Missed ESI 1 Patients": [len(missed_esi1)],
        "ESI 1 Recall": [
            (actual_esi1["Predicted ESI"] == 1).mean()
            if len(actual_esi1) > 0
            else np.nan
        ]
    })

    missed_breakdown = (
        missed_esi1["Predicted ESI"]
        .value_counts()
        .sort_index()
        .rename_axis("Incorrect Predicted ESI Level")
        .reset_index(name="Number of Missed ESI 1 Patients")
    )

    return summary, missed_breakdown


failure_summaries = []
missed_breakdowns = {}

for model_name, result in model_results.items():
    summary, missed = esi1_failure_summary(
        y_test, result["predictions"], model_name
    )
    failure_summaries.append(summary)
    missed_breakdowns[model_name] = missed

failure_summary = pd.concat(failure_summaries, ignore_index=True).round(3)

display(failure_summary)

In [ ]:
print(f"{best_complex_model}: where missed ESI Level 1 patients were placed")
display(missed_breakdowns[best_complex_model])

failure_summary.to_csv(
    DOC_DIR / "SOliver_Week7_ESI1_Failure_Summary.csv",
    index=False
)

print("Failure-mode summary saved.")

## 14. Side-by-Side Comparison Chart — All Five Models

A single chart summarising accuracy, macro F1, weighted F1 and ESI Level 1 recall across all five models (both Week 6 baselines and all three Week 7 complex-model candidates), for a quick visual gut-check before the numbers go into the memo.

In [ ]:
comparison_plot_data = (
    six_axis_benchmark
    .set_index("Model")[["Accuracy", "Macro F1", "Weighted F1", "ESI 1 Recall"]]
)

ax = comparison_plot_data.plot(kind="bar", figsize=(13, 6))

ax.set_title("Week 6 Baselines vs Week 7 Complex Models — Performance Comparison")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.set_ylim(0, 1)

plt.xticks(rotation=20, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()

plt.savefig(
    PLOT_DIR / "SOliver_Week7_Model_Comparison.png",
    dpi=300
)

plt.show()

## 15. Preliminary Compute-Cost Reflection (for Martina Griffith)

A first pass at the compute-cost angle Martina raised, across all five models. `assumed_daily_patients` and `retrains_per_year` are illustrative placeholders — replace them with real ED volume figures before this goes into the memo.

In [ ]:
assumed_daily_patients = 150   # placeholder — replace with real ED daily volume
retrains_per_year = 12         # placeholder — e.g. monthly retraining

cost_reflection_rows = []

for _, row in six_axis_benchmark.iterrows():
    daily_inference_seconds = (
        row["Inference Time — Single Row (ms/prediction)"] / 1000
        * assumed_daily_patients
    )
    annual_retraining_seconds = row["Training Time (s)"] * retrains_per_year

    cost_reflection_rows.append({
        "Model": row["Model"],
        "Assumed Daily Patients": assumed_daily_patients,
        "Est. Daily Inference Time (s)": round(daily_inference_seconds, 3),
        "Assumed Retrains / Year": retrains_per_year,
        "Est. Annual Retraining Time (s)": round(annual_retraining_seconds, 3),
    })

compute_cost_reflection = pd.DataFrame(cost_reflection_rows)

compute_cost_reflection.to_csv(
    DOC_DIR / "SOliver_Week7_Compute_Cost_Reflection.csv",
    index=False
)

display(compute_cost_reflection)

## 16. Summary and Next Steps

**What this notebook delivers:**

- Three trained complex-model candidates (Random Forest, XGBoost, LightGBM), all built on the exact Week 6 feature set and train/test split.
- Timing-instrumented retraining of both Week 6 baselines, so all five models are compared on equal footing.
- A five-model, six-axis quantitative benchmark table (`SOliver_Week7_Final_Benchmark_Table.csv`), saved to `week7_outputs/docs/`.
- An automatic best-of-three selection step (Section 10), so the interpretability, confusion matrix and failure-mode sections downstream focus on whichever complex model actually performs best rather than an arbitrarily chosen one.
- Confusion matrices and ESI Level 1 failure-mode breakdowns for every model.
- A compute-cost reflection across all five models, to seed the conversation with Martina Griffith.

**Still to do before this feeds the final memo:**

1. Run this notebook end-to-end with `xgboost` and `lightgbm` installed, and confirm `best_complex_model` picks the candidate you expect.
2. Sense-check the winning model's top feature importances with clinical reasoning (and Dr Reyes, if possible).
3. Add SHAP (or permutation importance on individual rows) so the interpretability claim in Section 11 is demonstrated, not just asserted.
4. Replace the placeholder values in Section 15 with real ED volume and retraining-cadence assumptions.
5. Update `docs/week-7-benchmark-table.md`, `docs/week-7-cost-benefit.md` / `.pdf` and `docs/decisions/2026-week-7-model-choice.md` to reflect the three-way complex-model comparison and whichever model Section 10 selects — these currently still describe the Random Forest-only comparison from before XGBoost and LightGBM were added.

In [ ]:
print("Generated plot files:")
for file in sorted(PLOT_DIR.glob("*")):
    print("-", file.name)

print("\nGenerated document files:")
for file in sorted(DOC_DIR.glob("*")):
    print("-", file.name)